# 00 · Груповий спліт (CPU)
**Вхід:** `train.csv` змагання. **Вихід:** `split.csv` (train ~100k + val 8 005) і `split_32k.csv` (train ~32k + та сама val). Час ~4 хв. **Запускати першим.**

**Навіщо:** у датасеті людський текст і його AI/fusion версії завжди лежать в одній частині train/test. Якщо ділити train випадково, версії одного тексту потрапляють і в train, і у валідацію, і F1 стає завищеним. Тому тексти об'єднуються в групи за спільними фразами з 6 слів (fusion це частково переписаний людський текст), а ділимо цілими групами.

**Обмеження:** AI-тексти не мають дослівних збігів з людськими, тому у групи не потрапляють (частка ai у групах з ≥2 текстів = 0). Група фактично це "людський текст + його fusion-версії", тож валідація для класу ai трохи оптимістична.

In [1]:
import glob, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.feature_extraction.text import HashingVectorizer
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

ID, TEXT, LABEL = "id", "text", "label"
SEED, N_SAMPLE, N_TRAIN = 42, 40000, 100000   # N_SAMPLE задає валідацію (як раніше), N_TRAIN розмір train
SH, KEEP, MAXBUCKET, MAXG = 6, 4, 20, 40   # довжина фрази; беремо 1/4 фраз; фрази, що є у >20 текстів, це шаблони; макс. розмір групи
OUT = "/kaggle/working"
train = pd.read_csv(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0])
N = len(train); print(train.shape)

(272352, 5)


## Фрази з 6 слів
Для кожного тексту беремо всі послідовності з 6 слів, хешуємо їх у числа й лишаємо кожну четверту (так швидше, а спільних фраз у версій одного тексту все одно багато).

In [2]:
hv = HashingVectorizer(analyzer="word", ngram_range=(SH, SH), n_features=2**30, alternate_sign=False,
                       binary=True, norm=None, token_pattern=r"(?u)\b\w+\b", dtype=np.float32)
H, D = [], []
for s in range(0, N, 10000):
    M = hv.transform(train[TEXT].iloc[s:s + 10000].str.slice(0, 20000))
    keep = (M.indices % KEEP) == 0
    rows = np.repeat(np.arange(M.shape[0]), np.diff(M.indptr)) + s
    H.append(M.indices[keep].astype(np.int64)); D.append(rows[keep].astype(np.int32))
H, D = np.concatenate(H), np.concatenate(D)
print("фраз-входжень:", len(H))

фраз-входжень: 36712699


## Пари текстів зі спільними фразами
Сортуємо за хешем фрази, прибираємо шаблонні фрази (що трапляються у >20 текстів), рахуємо для кожної пари текстів, скільки фраз вони ділять.

In [3]:
o = np.argsort(H, kind="stable"); H, D = H[o], D[o]
st = np.r_[0, np.flatnonzero(np.diff(H)) + 1]
cnt = np.diff(np.r_[st, len(H)])
c_per = np.repeat(cnt, cnt)
ok = (c_per >= 2) & (c_per <= MAXBUCKET)
H, D = H[ok], D[ok]

keys = []
for d in range(1, MAXBUCKET):
    m = H[:-d] == H[d:]
    if not m.any(): break
    a, b = D[:-d][m].astype(np.int64), D[d:][m].astype(np.int64)
    keys.append(np.minimum(a, b) * N + np.maximum(a, b))
u, shared = np.unique(np.concatenate(keys), return_counts=True)
pa, pb = u // N, u % N
print("пар зі спільними фразами:", len(u))

пар зі спільними фразами: 7934910


## Групи
Пари, що ділять щонайменше `ms` фраз, вважаємо версіями одного тексту. Піднімаємо `ms`, поки найбільша група не стане ≤ 40 (так відсікаємо випадкові збіги).

In [4]:
for ms in [3, 4, 6, 8, 12, 16, 24, 32]:
    k = shared >= ms
    A = coo_matrix((np.ones(k.sum()), (pa[k], pb[k])), shape=(N, N))
    ng, lab = connected_components(A, directed=False)
    mx = np.bincount(lab).max()
    print(f"мін. спільних фраз {ms}: груп {ng}, макс. розмір {mx}")
    if mx <= MAXG: break
train["g"] = lab

мін. спільних фраз 3: груп 136480, макс. розмір 64581
мін. спільних фраз 4: груп 168843, макс. розмір 7565
мін. спільних фраз 6: груп 185938, макс. розмір 409
мін. спільних фраз 8: груп 190045, макс. розмір 145
мін. спільних фраз 12: груп 193610, макс. розмір 101
мін. спільних фраз 16: груп 196145, макс. розмір 57
мін. спільних фраз 24: груп 200790, макс. розмір 43
мін. спільних фраз 32: груп 205581, макс. розмір 23


## Перевірка якості груп
Добре, якщо: середній розмір ближче до ~4, макс. ≤ 40, а таблиця нижче показує, що **fusion і human переважно в групах з ≥2 текстів**. Частка ai в групах із ≥2 текстів може бути меншою (AI-версія може не ділити дослівних фраз).

In [5]:
sz = train.groupby("g")[ID].transform("size")
gs = train.groupby("g").size()
nh = (train[LABEL] == "human").groupby(train["g"]).sum()
print("груп:", len(gs), "| середній розмір:", round(gs.mean(), 2), "| макс. розмір:", gs.max())
print("груп з >1 human:", round((nh > 1).mean(), 3))
print("частка текстів у групі з ≥2 текстів, по класах:")
print((sz > 1).groupby(train[LABEL]).mean().round(3))
print("по доменах:")
print((sz > 1).groupby(train["domain"]).mean().round(3))

груп: 205581 | середній розмір: 1.32 | макс. розмір: 23
груп з >1 human: 0.003
частка текстів у групі з ≥2 текстів, по класах:
label
ai        0.000
fusion    0.798
human     0.693
Name: id, dtype: float64
по доменах:
domain
arxiv_papers              0.130
blog_authorship_corpus    0.241
bundestag                 0.561
cnn_news                  0.465
euro_court_cases          0.517
gutenberg                 0.059
house_of_commons          0.377
student_essays            0.458
Name: id, dtype: float64


## Спліт цілими групами
Валідація: перші 20% груп із вибірки ~40k текстів (8 005 текстів). Train: `split_32k.csv` це решта тієї ж вибірки (~32k), `split.csv` це до 100k текстів з усіх інших груп. Валідація в обох файлах та сама, тому результати моделей на 32k і на 100k порівнювані.

In [6]:
rng = np.random.default_rng(SEED)
gids = rng.permutation(gs.index.values)
cum = np.cumsum(gs.loc[gids].values)
chosen = gids[:np.searchsorted(cum, N_SAMPLE) + 1]          # ті самі 40k, що й раніше
va_g = set(chosen[:int(0.2 * len(chosen))])                 # валідація НЕ змінюється (~8k)
rest = np.array([g for g in gids if g not in va_g])         # train: усі інші групи до N_TRAIN текстів
cum2 = np.cumsum(gs.loc[rest].values)
tr_g = set(rest[:np.searchsorted(cum2, N_TRAIN) + 1])
sub = train[train["g"].isin(va_g | tr_g)].copy()
sub["part"] = np.where(sub["g"].isin(va_g), "va", "tr")
print(sub["part"].value_counts().to_dict(), "(val має бути 8005, як і раніше)")
print(pd.crosstab(sub["part"], sub[LABEL], normalize="index").round(3))
print(pd.crosstab(sub["part"], sub["domain"], normalize="index").round(3))
assert not (set(sub[sub.part == "tr"]["g"]) & set(sub[sub.part == "va"]["g"]))
sub[[ID, "part"]].to_csv(f"{OUT}/split.csv", index=False)

# менший train (~32k) для експериментів TF-IDF / перплексія / енкодер 32k: та сама валідація
chosen_tr = set(chosen[int(0.2 * len(chosen)):])
sub32 = train[train["g"].isin(va_g | chosen_tr)].copy()
sub32["part"] = np.where(sub32["g"].isin(va_g), "va", "tr")
print("split_32k:", sub32["part"].value_counts().to_dict())
sub32[[ID, "part"]].to_csv(f"{OUT}/split_32k.csv", index=False)

{'tr': 100002, 'va': 8005} (val має бути 8005, як і раніше)
label     ai  fusion  human
part                       
tr     0.449   0.306  0.245
va     0.444   0.312  0.244
domain  arxiv_papers  blog_authorship_corpus  bundestag  cnn_news  \
part                                                                
tr             0.052                   0.149      0.193     0.163   
va             0.053                   0.147      0.190     0.166   

domain  euro_court_cases  gutenberg  house_of_commons  student_essays  
part                                                                   
tr                 0.101      0.028             0.116           0.198  
va                 0.103      0.025             0.119           0.197  
split_32k: {'tr': 31995, 'va': 8005}
